In [1]:
# ==============================================
# CELL 1: Install & Imports
# ==============================================
import os, sys, shutil, glob, yaml, time
from ultralytics import YOLO
print("✅ Imports ready")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Imports ready


In [2]:
# ==============================================
# CELL 2: Configuration
# ==============================================
# ⚙️ CHANGE THIS to match your Kaggle dataset name
DATASET_DIR = "/kaggle/input//datasets/naofunyannn/yolov11gtsdb"
WORKING = "/kaggle/working"
PROJECT = os.path.join(WORKING, "runs", "detect")
NAME = "gtsdb_traffic_sign"
# Training settings for 2x T4
BASE_MODEL = "yolo11s.pt"
EPOCHS = 150
BATCH_SIZE = 16       # 8 per GPU × 2
IMG_SIZE = 1280
PATIENCE = 30
SAVE_PERIOD = 5       # Checkpoint every 5 epochs
DEVICE = "0,1"        # 2x T4
print("✅ Config set")

✅ Config set


In [3]:
# ==============================================
# CELL 3: Find & validate dataset
# ==============================================
# Find data.yaml (might be in root or subfolder)
yaml_candidates = glob.glob(os.path.join(DATASET_DIR, "**", "data.yaml"), recursive=True)
yaml_candidates += glob.glob(os.path.join(DATASET_DIR, "data.yaml"))
if not yaml_candidates:
    print("❌ data.yaml not found! Available files in input:")
    for f in glob.glob(os.path.join(DATASET_DIR, "**", "*"), recursive=True)[:20]:
        print(f"   {f}")
    raise FileNotFoundError("data.yaml not found")
DATA_YAML = yaml_candidates[0]
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)
print(f"✅ Found: {DATA_YAML}")
print(f"   Classes: {data_config.get('nc')} → {data_config.get('names')}")
# Count images
data_root = os.path.dirname(DATA_YAML)
for split in ['train', 'valid', 'val', 'test']:
    img_dir = os.path.join(data_root, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"   {split}: {count} images")

✅ Found: /kaggle/input//datasets/naofunyannn/yolov11gtsdb/data.yaml
   Classes: 1 → ['sign']
   train: 1489 images
   valid: 270 images


In [4]:
# ==============================================
# CELL 4: Merge all classes → single "traffic_sign"
# ==============================================
# Skip this cell if your dataset is already single-class
MERGED_DIR = os.path.join(WORKING, "GTSDB_merged")
MERGED_YAML = os.path.join(MERGED_DIR, "data.yaml")
if os.path.exists(MERGED_YAML):
    print(f"✅ Merged dataset already exists. Reusing.")
else:
    data_root = os.path.dirname(DATA_YAML)
    print(f"Copying dataset to {MERGED_DIR}...")
    shutil.copytree(data_root, MERGED_DIR, dirs_exist_ok=True)
    # Rewrite all label files: set class ID to 0
    modified = 0
    for split in ['train', 'valid', 'val', 'test']:
        labels_dir = os.path.join(MERGED_DIR, split, 'labels')
        if not os.path.exists(labels_dir):
            continue
        for lf in glob.glob(os.path.join(labels_dir, '*.txt')):
            with open(lf, 'r') as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = '0'
                    new_lines.append(' '.join(parts) + '\n')
            with open(lf, 'w') as f:
                f.writelines(new_lines)
            modified += 1
    # Update data.yaml
    with open(MERGED_YAML, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['nc'] = 1
    cfg['names'] = ['traffic_sign']
    for key in ['train', 'val', 'valid', 'test']:
        if key in cfg and '..' in str(cfg[key]):
            cfg[key] = str(cfg[key]).replace('..', '.')
    with open(MERGED_YAML, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"✅ Merged {modified} label files → single class 'traffic_sign'")
# This is the data.yaml to use for training
ACTIVE_YAML = MERGED_YAML
print(f"Using: {ACTIVE_YAML}")

Copying dataset to /kaggle/working/GTSDB_merged...
✅ Merged 1759 label files → single class 'traffic_sign'
Using: /kaggle/working/GTSDB_merged/data.yaml


In [5]:
# ==============================================
# CELL 5: Train (with auto-resume)
# ==============================================
# If Kaggle times out, just re-run THIS CELL.
# It will resume from the last checkpoint automatically.
last_pt = os.path.join(PROJECT, NAME, "weights", "last.pt")
if os.path.exists(last_pt):
    print(f"🔄 RESUMING from: {last_pt}")
    model = YOLO(last_pt)
    results = model.train(resume=True)
else:
    print("🆕 Starting fresh training")
    model = YOLO(BASE_MODEL)
    results = model.train(
        data=ACTIVE_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        project=PROJECT,
        name=NAME,
        exist_ok=True,
        device=DEVICE,
        # Optimizer
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=5,
        # Augmentation
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=10.0, translate=0.1, scale=0.5,
        shear=5.0, perspective=0.001,
        flipud=0.0, fliplr=0.0,  # No flips!
        mosaic=1.0, mixup=0.1, erasing=0.2,
        # Checkpoints
        save=True,
        save_period=SAVE_PERIOD,
        plots=True,
        verbose=True,
    )
best_pt = os.path.join(PROJECT, NAME, "weights", "best.pt")
print(f"\n✅ Training done! Best: {best_pt}")

🆕 Starting fresh training
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/GTSDB_merged/data.yaml, degrees=10.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.2, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s.pt, momentum=

In [6]:
# ==============================================
# CELL 6: Validate
# ==============================================
best_pt = os.path.join(PROJECT, NAME, "weights", "best.pt")
model = YOLO(best_pt)
metrics = model.val(
    data=ACTIVE_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
)
print(f"\n📊 Results:")
print(f"   mAP50:     {metrics.box.map50:.4f}")
print(f"   mAP50-95:  {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall:    {metrics.box.mr:.4f}")

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2175.2±773.0 MB/s, size: 142.2 KB)
val: Scanning /kaggle/working/GTSDB_merged/valid/labels.cache... 270 images, 56 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 270/270 59.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 2.0it/s 8.4s
                   all        270        348      0.984      0.945      0.979      0.806
Speed: 2.5ms preprocess, 22.5ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val

📊 Results:
   mAP50:     0.9794
   mAP50-95:  0.8063
   Precision: 0.9836
   Recall:    0.9454


In [7]:
# ==============================================
# CELL 7: Save model for download
# ==============================================
output_dir = os.path.join(WORKING, "trained_model")
os.makedirs(output_dir, exist_ok=True)
best_pt = os.path.join(PROJECT, NAME, "weights", "best.pt")
last_pt = os.path.join(PROJECT, NAME, "weights", "last.pt")
shutil.copy2(best_pt, os.path.join(output_dir, "gtsdb_best.pt"))
if os.path.exists(last_pt):
    shutil.copy2(last_pt, os.path.join(output_dir, "gtsdb_last.pt"))
print(f"✅ Models saved to: {output_dir}/")
print(f"\n📥 Download from Kaggle Output tab:")
print(f"   → gtsdb_best.pt")
print(f"\nThen in your project:")
print(f'   DETECTOR_PATH = "yolo11_model/gtsdb_best.pt"')

✅ Models saved to: /kaggle/working/trained_model/

📥 Download from Kaggle Output tab:
   → gtsdb_best.pt

Then in your project:
   DETECTOR_PATH = "yolo11_model/gtsdb_best.pt"
